[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/translations/ru/notebooks/ch03_mdp_bellman_ru.ipynb)

<div style="background: linear-gradient(135deg, #0a1628 0%, #1a2a4a 50%, #2a3f6f 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #a78bfa; font-size: 2.2em; margin: 0 0 10px 0;">Глава 3. Марковские процессы принятия решений и уравнения Беллмана</h1>
  <p style="color: #a8dadc; font-size: 1.1em; margin: 0;">Определяем студенческий MDP, считаем матрицы переходов, применяем оценивание Беллмана и выводим оптимальную стратегию.</p>
</div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #4fc3f7;">
  <strong style="color: #4fc3f7;">Подготовка окружения</strong><br>
  <span style="color: #cdd9e5;">Только NumPy — GPU не нужен. Время выполнения менее 5 секунд.</span>
</div>

In [ ]:
!pip install numpy matplotlib --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import itertools

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
})
print('Setup complete.')

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #a78bfa;">
  <strong style="color: #a78bfa;">1. Формализм MDP</strong>
</div>

**Марковский процесс принятия решений** (MDP) — это пятёрка $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:
- $\mathcal{S}$ — пространство состояний
- $\mathcal{A}$ — пространство действий
- $P(s'|s,a)$ — вероятность перехода
- $R(s,a)$ — ожидаемое вознаграждение
- $\gamma \in [0,1)$ — коэффициент дисконтирования

**Студенческий MDP** (по Sutton \& Barto, пример 3.3): студент переключается между учёбой и развлечениями.

Состояния: `Class1`, `Class2`, `Class3`, `Рутуб`, `Pub`, `Pass`, `Sleep`.  
Используем упрощённую версию из 5 состояний: **C1, C2, C3, FB, Sleep** с двумя действиями в каждом состоянии.

In [ ]:
# ── Student MDP Definition ──
# States: 0=C1, 1=C2, 2=C3, 3=Рутуб, 4=Sleep(terminal)
# Actions: 0=Study, 1=Slack

STATES  = ['C1', 'C2', 'C3', 'Рутуб', 'Sleep']
ACTIONS = ['Study', 'Slack']
N_S, N_A = len(STATES), len(ACTIONS)
GAMMA = 0.9

# P[s, a, s'] = probability of transitioning from s to s' under action a
P = np.zeros((N_S, N_A, N_S))
# R[s, a] = immediate reward
R = np.zeros((N_S, N_A))

# --- Transitions (Study action) ---
P[0, 0, 1] = 1.0   # C1 + Study -> C2
P[1, 0, 2] = 1.0   # C2 + Study -> C3
P[2, 0, 4] = 1.0   # C3 + Study -> Sleep (Pass!)
P[3, 0, 0] = 1.0   # FB + Study -> C1
P[4, 0, 4] = 1.0   # Sleep stays

# --- Transitions (Slack action) ---
P[0, 1, 3] = 1.0   # C1 + Slack -> Рутуб
P[1, 1, 1] = 0.5; P[1, 1, 3] = 0.5   # C2 + Slack -> C2 or FB
P[2, 1, 4] = 0.6; P[2, 1, 2] = 0.4   # C3 + Slack -> Sleep(Fail) or stay
P[3, 1, 3] = 1.0   # FB + Slack -> FB
P[4, 1, 4] = 1.0   # Sleep stays

# --- Rewards ---
R[0, 0] = -2  # C1, Study
R[1, 0] = -2  # C2, Study
R[2, 0] = +10 # C3, Study -> Pass!
R[3, 0] = -1  # FB, Study (come back)
R[0, 1] = -1  # C1, Slack
R[1, 1] = -1  # C2, Slack
R[2, 1] = -3  # C3, Slack -> risky
R[3, 1] = +1  # FB, Slack -> enjoyable

for s, a in itertools.product(range(N_S), range(N_A)):
    total = P[s, a].sum()
    assert abs(total - 1.0) < 1e-9, f'Transition sums to {total} for s={s}, a={a}'

print('MDP defined. States:', STATES)
print('Reward matrix R[s,a]:')
print(np.array([[f'{STATES[s]}+{ACTIONS[a]}={R[s,a]:+.0f}' for a in range(N_A)] for s in range(N_S)]))

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #a78bfa;">
  <strong style="color: #a78bfa;">2. Уравнение Беллмана для оценивания (Policy Evaluation)</strong>
</div>

При фиксированной стратегии $\pi$ **функция ценности состояний** $V^\pi(s)$ удовлетворяет:
$$V^\pi(s) = \sum_a \pi(a|s) \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V^\pi(s') \right]$$

Это система $|\mathcal{S}|$ линейных уравнений, решаемая итеративно или обращением матрицы.

In [ ]:
# ── Iterative Policy Evaluation ──

def policy_evaluation(policy, P, R, gamma=0.9, theta=1e-8, max_iter=1000):
    """Iteratively evaluate a policy.
    policy: array of shape (N_S,) giving action index for each state.
    Returns V and convergence history.
    """
    n_s = P.shape[0]
    V = np.zeros(n_s)
    history = [V.copy()]

    for _ in range(max_iter):
        V_new = np.zeros(n_s)
        for s in range(n_s):
            a = policy[s]
            V_new[s] = R[s, a] + gamma * np.dot(P[s, a], V)
        history.append(V_new.copy())
        if np.max(np.abs(V_new - V)) < theta:
            V = V_new
            break
        V = V_new

    return V, history


# Policy 1: always Study
study_policy = np.zeros(N_S, dtype=int)   # Action 0 = Study

# Policy 2: always Slack
slack_policy = np.ones(N_S, dtype=int)    # Action 1 = Slack

V_study, hist_study = policy_evaluation(study_policy, P, R, GAMMA)
V_slack, hist_slack = policy_evaluation(slack_policy, P, R, GAMMA)

print('Value function — Always Study:')
for s, v in zip(STATES, V_study):
    print(f'  V({s:8s}) = {v:7.3f}')
print('\nValue function — Always Slack:')
for s, v in zip(STATES, V_slack):
    print(f'  V({s:8s}) = {v:7.3f}')

In [ ]:
# ── Plot: Value Functions & Convergence ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart: V values for both policies
ax = axes[0]
x = np.arange(N_S)
w = 0.35
bars1 = ax.bar(x - w/2, V_study, width=w, color='#56d364', label='Always Study', alpha=0.85)
bars2 = ax.bar(x + w/2, V_slack, width=w, color='#e94560', label='Always Slack', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(STATES)
ax.set_title('State Value Functions', fontsize=13)
ax.set_ylabel('V(s)')
ax.legend()
ax.axhline(0, color='#8b949e', linestyle='--', alpha=0.4)
ax.grid(True, alpha=0.3, axis='y')

# Convergence: max |ΔV| per iteration
ax = axes[1]
def convergence_deltas(history):
    return [np.max(np.abs(history[i+1] - history[i])) for i in range(len(history)-1)]

d_study = convergence_deltas(hist_study)
d_slack = convergence_deltas(hist_slack)
ax.semilogy(d_study, color='#56d364', linewidth=2, label='Always Study')
ax.semilogy(d_slack, color='#e94560', linewidth=2, label='Always Slack')
ax.set_title('Policy Evaluation Convergence', fontsize=13)
ax.set_xlabel('Iteration')
ax.set_ylabel('max|ΔV| (log scale)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Student MDP — Policy Evaluation', fontsize=14)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #a78bfa;">
  <strong style="color: #a78bfa;">3. Уравнение оптимальности Беллмана и оптимальная стратегия</strong>
</div>

**Оптимальная функция ценности** $V^*(s)$ удовлетворяет:
$$V^*(s) = \max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V^*(s') \right]$$

Решаем итерациями оператора оптимальности Беллмана до сходимости (предварительный показ итерации ценности — полностью в главе 4).

In [ ]:
# ── Value Iteration (simplified) ──

def value_iteration(P, R, gamma=0.9, theta=1e-8):
    n_s, n_a = R.shape
    V = np.zeros(n_s)
    for _ in range(1000):
        Q = np.array([R[:, a] + gamma * P[:, a, :] @ V for a in range(n_a)]).T  # (n_s, n_a)
        V_new = Q.max(axis=1)
        if np.max(np.abs(V_new - V)) < theta:
            V = V_new
            break
        V = V_new
    optimal_policy = Q.argmax(axis=1)
    return V, optimal_policy, Q

V_star, pi_star, Q_star = value_iteration(P, R, GAMMA)

print('Optimal Value Function V*(s):')
for s, v in zip(STATES, V_star):
    print(f'  V*({s:8s}) = {v:7.3f}')
print('\nOptimal Policy π*(s):')
for s, a in zip(STATES, pi_star):
    print(f'  π*({s:8s}) = {ACTIONS[a]}')

In [ ]:
# ── Plot: Q-values and optimal policy ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heatmap of Q*(s,a)
ax = axes[0]
im = ax.imshow(Q_star, aspect='auto', cmap='plasma')
ax.set_xticks(range(N_A))
ax.set_xticklabels(ACTIONS)
ax.set_yticks(range(N_S))
ax.set_yticklabels(STATES)
for s in range(N_S):
    for a in range(N_A):
        ax.text(a, s, f'{Q_star[s,a]:.1f}', ha='center', va='center',
                color='white', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, label='Q*(s,a)')
ax.set_title('Optimal Action-Value Q*(s,a)', fontsize=13)

# Policy comparison: all three policies
ax = axes[1]
policy_labels = ['Always\nStudy', 'Always\nSlack', 'Optimal\nπ*']
value_profiles = [V_study, V_slack, V_star]
policy_colors  = ['#56d364', '#e94560', '#ffd700']
x = np.arange(N_S)
w = 0.25
for i, (label, V, color) in enumerate(zip(policy_labels, value_profiles, policy_colors)):
    ax.bar(x + (i - 1) * w, V, width=w, color=color, label=label, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(STATES)
ax.set_title('Value Functions: Policy Comparison', fontsize=13)
ax.set_ylabel('V(s)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='#8b949e', linestyle='--', alpha=0.4)

plt.suptitle('Student MDP — Optimal Policy via Bellman Optimality', fontsize=14)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #a78bfa;">
  <strong style="color: #a78bfa;">4. Визуализация студенческого MDP в виде графа</strong>
</div>

In [ ]:
# ── MDP Diagram ──
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_facecolor('#0d1117')

# Node positions
pos = {
    'C1':       (1, 2),
    'C2':       (3, 2),
    'C3':       (5, 2),
    'Рутуб': (2, 0.2),
    'Sleep':    (7, 2),
}
node_colors = {
    'C1': '#1f6feb', 'C2': '#1f6feb', 'C3': '#1f6feb',
    'Рутуб': '#e94560', 'Sleep': '#2ea043'
}

for name, (x, y) in pos.items():
    circle = plt.Circle((x, y), 0.45, color=node_colors[name], zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, name, ha='center', va='center',
            fontsize=8, color='white', fontweight='bold', zorder=4)

# Edges (simplified — Study action only)
edges_study = [('C1','C2',-2), ('C2','C3',-2), ('C3','Sleep',+10), ('Рутуб','C1',-1)]
for src, dst, r in edges_study:
    x1, y1 = pos[src]; x2, y2 = pos[dst]
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#56d364', lw=2))
    mx, my = (x1+x2)/2, (y1+y2)/2 + 0.25
    ax.text(mx, my, f'Study\nr={r}', ha='center', va='bottom',
            fontsize=7.5, color='#56d364')

# Slack edges
edges_slack = [('C1','Рутуб',-1), ('C2','Рутуб',-1), ('Рутуб','Рутуб',+1)]
for src, dst, r in edges_slack:
    x1, y1 = pos[src]; x2, y2 = pos[dst]
    if src == dst:   # Self-loop
        ax.annotate('', xy=(x1+0.5, y1-0.2), xytext=(x1-0.5, y1-0.2),
                    arrowprops=dict(arrowstyle='->', color='#e94560',
                                   connectionstyle='arc3,rad=-0.8', lw=2))
        ax.text(x1, y1-0.8, f'Slack r={r}', ha='center', fontsize=7.5, color='#e94560')
    else:
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', color='#e94560',
                                   connectionstyle='arc3,rad=0.2', lw=2))
        mx, my = (x1+x2)/2 - 0.3, (y1+y2)/2
        ax.text(mx, my, f'Slack\nr={r}', ha='center', fontsize=7.5, color='#e94560')

ax.set_xlim(-0.2, 8.5)
ax.set_ylim(-1.2, 3.2)
ax.set_title('Student MDP — State Transition Diagram', color='#c9d1d9', fontsize=13)
ax.axis('off')

study_patch = mpatches.Patch(color='#56d364', label='Study action')
slack_patch = mpatches.Patch(color='#e94560', label='Slack action')
ax.legend(handles=[study_patch, slack_patch], loc='lower right')
plt.tight_layout()
plt.show()

<div style="background: #0d2137; padding: 20px; border-radius: 8px; border: 1px solid #1f6feb; margin-top: 20px;">
  <h3 style="color: #79c0ff; margin-top: 0;">Глава 3 — ключевые выводы</h3>
  <ul style="color: #c9d1d9; line-height: 1.8;">
    <li><strong>MDP</strong> формализует задачи последовательного принятия решений со свойством Маркова.</li>
    <li><strong>Оценивание стратегии</strong> решает уравнение Беллмана для нахождения $V^\pi$ итеративно.</li>
    <li><strong>Уравнение оптимальности Беллмана</strong> характеризует $V^*$ — максимально достижимую ценность.</li>
    <li>Оптимальную стратегию $\pi^*$ можно жадно извлечь из $V^*$ или $Q^*$.</li>
    <li>В студенческом MDP учиться оптимально из C3 (большое вознаграждение), но неоптимально из <<Рутуба>> (большая цена).</li>
  </ul>
</div>